# Unified platform: House of Hope / Lighthouse — system design at a glance

**Audience:** Executives, product, and engineering  
**Goal:** Explain how donor funding, resident outcomes, partners, and social outreach form **one** operational and analytical system — not four silos.

This notebook ties together the same entities your app and CSV exports use: `supporters`, `donations`, `donation_allocations`, `safehouses`, `residents`, `partners` / `partner_assignments`, `social_media_posts`, `safehouse_monthly_metrics`, and `public_impact_snapshots`.

---

## The one-sentence story

**Mission delivery is funded by relationships** (supporters → gifts), **routed to places and programs** (allocations → safehouses), **executed with local partners and staff** (assignments + case records), **measured per site and over time** (monthly metrics), **communicated outward** (social + public impact snapshots), and **fed back into strategy** (which channels and programs to scale).

## 1. Four domains, one platform

| Domain | What it is in this system | Primary data anchors |
|--------|---------------------------|----------------------|
| **Donor funding** | Who gives, how, and why | `supporters`, `donations`, `donation_allocations`, `in_kind_donation_items` |
| **Resident outcomes** | Care, risk, education, health, incidents | `residents`, `process_recordings`, `education_records`, `health_wellbeing_records`, `intervention_plans`, `incident_reports`, `home_visitations` |
| **Partner involvement** | Contracted / aligned orgs tied to sites and program areas | `partners`, `partner_assignments` |
| **Social media outreach** | Content, engagement, attribution to gifts | `social_media_posts` (`referral_post_id` on `donations`) |

The **unifier** is the **safehouse** (and program area): money is allocated to it, residents live there, partners are assigned to it, and monthly rollups describe how it is performing.

## 2. How money flows: supporter → donation → allocation → safehouse → resident impact

**Conceptual chain**

1. **Supporter** (`supporters`) — identity, segment, acquisition channel (e.g. SocialMedia, Event), relationship to the mission.
2. **Donation** (`donations`) — monetary or in-kind (or time), currency, campaign, channel; optional **`referral_post_id`** linking a gift to a **social post**.
3. **Allocation** (`donation_allocations`) — splits a donation across **`safehouse_id`** + **`program_area`** (Education, Transport, Wellbeing, Operations, …) with `amount_allocated` and date.
4. **Safehouse** (`safehouses`) — the operational unit that spends and reports; ties to **residents** via `residents.safehouse_id`.
5. **Resident impact** — not always a direct FK from “pesos → child” in one row; impact is **observed** through longitudinal tables (education progress, health scores, incidents, reintegration status) and **aggregated** in `safehouse_monthly_metrics` and `public_impact_snapshots`.

```mermaid
flowchart LR
  S[supporters] --> D[donations]
  SM[social_media_posts] -. referral_post_id .-> D
  D --> A[donation_allocations]
  A --> SH[safehouses]
  SH --> R[residents]
  R --> O[outcome tables]
  O --> M[safehouse_monthly_metrics]
  M --> P[public_impact_snapshots]
```

**Why this matters for product:** fundraising isn’t “finished” at payment capture. The platform’s job is to **close the loop**: show supporters and executives *where* money landed and *what changed* in outcomes — without breaking anonymity rules for public reporting.

In [ ]:
# Optional: quantify the financial chain against local CSV exports
from pathlib import Path

import pandas as pd

_csv = Path("ml-pipelines/lighthouse_csv_v7")
DATA_DIR = _csv if _csv.is_dir() else Path("lighthouse_csv_v7")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Missing {DATA_DIR.resolve()} — open this notebook from repo root or the ml-pipelines folder."
    )

supporters = pd.read_csv(DATA_DIR / "supporters.csv")
donations = pd.read_csv(DATA_DIR / "donations.csv", parse_dates=["donation_date"])
allocations = pd.read_csv(DATA_DIR / "donation_allocations.csv", parse_dates=["allocation_date"])
safehouses = pd.read_csv(DATA_DIR / "safehouses.csv")
residents = pd.read_csv(DATA_DIR / "residents.csv")

chain = (
    donations.merge(supporters, on="supporter_id", how="left", suffixes=("", "_sup"))
    .merge(allocations, on="donation_id", how="inner")
    .merge(safehouses, on="safehouse_id", how="left", suffixes=("", "_sh"))
)

display(
    pd.Series(
        {
            "supporters": len(supporters),
            "donations": len(donations),
            "allocation rows": len(allocations),
            "joined supporter→donation→allocation→safehouse rows": len(chain),
            "safehouses": len(safehouses),
            "residents": len(residents),
            "donations with social referral_post_id": donations["referral_post_id"].notna().sum(),
        }
    ).to_frame("count")
)

amt = pd.to_numeric(chain["amount_allocated"], errors="coerce")
print("Sample: total allocated (PHP) in joined chain:", float(amt.sum()))
chain[["display_name", "donation_id", "safehouse_id", "program_area", "amount_allocated"]].head(8)

## 3. How data flows: operations → metrics → public impact

**Operations (ground truth)**  
Case managers, clinicians, educators, and house leads produce **events and states**: visits, sessions, plans, incidents, enrollment, health checks. These are the “atoms” of accountability.

**Metrics (roll-up layer)**  
`safehouse_monthly_metrics` turns those atoms into **comparable time series** per site: active residents, average education progress, average health score, counts of recordings / visitations / incidents. This is the **internal management** view — still sensitive, but structured.

**Public impact (narrative + aggregates)**  
`public_impact_snapshots` is the **stewardship layer**: headline, summary text, optional `metric_payload_json`, publish flags. It is where you enforce **anonymization**, messaging, and what executives are willing to say in public.

```mermaid
flowchart TB
  subgraph ops [Operations — granular]
    PR[process_recordings]
    HV[home_visitations]
    ER[education_records]
    HW[health_wellbeing_records]
    IP[intervention_plans]
    IR[incident_reports]
  end
  subgraph roll [Metrics — periodic]
    MM[safehouse_monthly_metrics]
  end
  subgraph pub [Public impact — governed]
    PI[public_impact_snapshots]
  end
  ops --> MM
  MM --> PI
```

**Product implication:** the platform needs **three visibility modes** — staff detail, leadership aggregates, and public-safe storytelling — all derived from the same lineage so numbers don’t contradict each other.

## 4. How insights in one domain change another

| If you learn … | You can act in … | Mechanism in data |
|----------------|------------------|-------------------|
| Certain **post types** or **campaigns** drive gifts | **Fundraising** and **content calendar** | `social_media_posts` engagement + `donations.referral_post_id` |
| A **safehouse** has rising **incidents** or stalled **education progress** | **Allocations** and **partner assignments** | `safehouse_monthly_metrics` + `partner_assignments.program_area` |
| **Program_area** spend doesn’t correlate with outcome movement | **Finance + programs** | `donation_allocations.program_area` vs education/health trends |
| Donors acquired via **SocialMedia** behave differently | **Retention journeys** | `supporters.acquisition_channel` × repeat `donations` |
| Public snapshots **understate** real progress | **Comms + measurement definitions** | Compare `metric_payload_json` to internal `safehouse_monthly_metrics` rules |

**Unified insight:** the system is a **feedback graph**. Social proves *attention*, donations prove *commitment*, allocations prove *deployment*, operational tables prove *execution*, metrics prove *trend*, public impact proves *trust* with the outside world. Weakness anywhere shows up as a broken link (e.g. gifts with no allocation story, or metrics that never become publishable impact).

## 5. A unified executive dashboard (suggested)

**North-star layout — single page, four quadrants + center**

| Zone | Question it answers | Example widgets |
|------|---------------------|-----------------|
| **Center** | Are we solvent and mission-aligned this quarter? | Cash-like inflow (PHP), recurring vs one-time, top 5 campaigns, variance vs plan |
| **Top-left — Funding** | Who gives and through what channel? | Acquisition mix, donor concentration, social-attributed revenue, in-kind value |
| **Top-right — Outcomes** | Are girls safer and progressing? | Active residents, risk distribution, education/health trajectory, incident rate |
| **Bottom-left — Partners** | Is external capacity where we need it? | Assignments by safehouse + program_area, coverage gaps, partner tenure |
| **Bottom-right — Reach** | Is our story working? | Impressions/reach, engagement, referral conversion, CTA performance |

**Cross-links (drill-through, not separate reports)**

- Click a **safehouse** → allocations received + monthly metrics trend + open partner assignments + resident counts (aggregated).
- Click a **campaign** → donations + attributed posts + allocation mix.
- Click a **public snapshot month** → underlying internal metric definitions used (read-only for execs; editable for comms with permissions).

**Guardrails:** role-based masking, audit trail on snapshot publishing, and explicit “public vs internal” labels on every KPI.

---

## 6. As a real SaaS product

**What you’re selling** is not “a donor portal” or “a case management app” alone — it is **Mission Operations Cloud**: multi-tenant software for faith-based and secular nonprofits running **residential + community programs** with **distributed partners**, **regulated storytelling**, and **donor-grade transparency**.

**Core modules (SKUs or tiers)**

1. **CareOps** — residents, incidents, interventions, education/health records, visitations (workflow + compliance).
2. **FundOps** — CRM-light supporters, gifts, in-kind, allocations to sites/programs, recognition.
3. **PartnerNet** — partner directory, assignments, SLAs, document vault, joint outcomes.
4. **Reach & Attribution** — content calendar, UTM/referral capture, modeled social → donation lift.
5. **Impact Studio** — metric definitions, snapshot builder, approval workflow, embeddable public widgets.
6. **Executive Fabric** — the unified dashboard above + exports + board packs.

**Platform primitives**

- **Tenant** = organization; **Sites** = safehouses / hubs; **Programs** = orthogonal to finance (`program_area`).
- **Lineage service:** every public number links to internal definitions (versioned).
- **Consent & anonymization layer:** rules for what can appear in Reach vs Impact Studio.
- **API + CSV + SQLite/warehouse** for analysts (your notebooks stay first-class).

**Moat:** competitors do CRM or case notes; few connect **allocation-level finance** to **longitudinal resident outcomes** to **partner accountability** to **attributed acquisition** with **governed public narrative** in one schema and UX.

## 7. Three killer features that would make the platform stand out

### 1. **Impact lineage & “explain this number” (trust automation)**
Any KPI on the exec dashboard or a public snapshot can be expanded into a **trace**: which donations rolled into which allocations, which safehouse metrics rolled into the month, which anonymization rule was applied. **Why killer:** boards and major donors increasingly demand defensible impact; this turns governance into a product feature, not a spreadsheet hunt.

### 2. **Partner–program–finance triangle**
A live view that joins **`partner_assignments`** (who is responsible for what program area at which site) with **`donation_allocations`** and outcome deltas from **`safehouse_monthly_metrics`**. Alerts when a site is **funded for Education** but **education progress** stalls while **incidents** rise — suggesting partner capacity or care model issues, not just “fundraiser story.” **Why killer:** nonprofits die from silent operational drift; this makes responsibility and capital visible together.

### 3. **Closed-loop growth: content → gift → allocation → outcome → next best content**
Use `referral_post_id` and channel metadata to build **feedback loops**: which narratives actually funded which program areas, and which program areas showed the strongest outcome response — then recommend **next campaigns** and **creative angles** with guardrails (ethics panel, no exploitative storytelling). **Why killer:** marketing stops guessing; programs stop flying blind on what stories resonate *and* work.

---

### Big-picture takeaway

Treat **donors, residents, partners, and audience** as nodes on one graph anchored by **sites, time, and program area**. Your current CSV/app model already encodes that graph; the product opportunity is to **surface it**, **govern it**, and **close the loops** so every department is steering the same ship.